### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="bad_customer_detection",
    dataset_year="2020",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/podsyp/is-this-a-good-customer",
    download_description="""
Download the aug_train.csv from https://www.kaggle.com/datasets/podsyp/is-this-a-good-customer.

Create a folder and move the .csv file there: mkdir -p local-data-warehouse/bad_customer_detection
""",
    # References
    academic_reference_bibtex="""@misc{Podsyp2020IsThisAGoodCustomer,
  title={Is This a Good Customer?},
  author={Podsyp},
  year={2020},
  publisher={Kaggle},
  url={https://www.kaggle.com/datasets/podsyp/is-this-a-good-customer}
}
""",
    academic_reference_bibtex_key="Podsyp2020IsThisAGoodCustomer",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
        - We shuffle the data as it has an original distribution shift.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="bad_customer",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="bad_customer",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "clients.csv.xls")

target_feature = "bad_customer"
df.rename(columns={"bad_client_target": target_feature}, inplace=True)
df[target_feature] = df[target_feature].map({0: "No", 1: "Yes"}).astype("category")

# Data is ordered, thus dist shift for original order. Shuffling the data removes this.
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

cat_cols = ['month', 'sex', 'education', 'product_type', 'having_children_flg', 'region', 'family_status', 'phone_operator', 'is_client',]
df[cat_cols] = df[cat_cols].astype("category")

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,723
Columns: 14
Use sampling: False (sample size: 1,723)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['credit_amount', 'income', 'age', 'product_type', 'credit_term', 'month', 'education', 'phone_operator', 'region', 'family_status']
Rows remaining as candidates after top-10 filter: 0 (of 1,723)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,month,credit_amount,credit_term,age,sex,education,product_type,having_children_flg,region,income,family_status,phone_operator,is_client,bad_customer
0,3,9500,9,22,female,Secondary special education,Computers,0,2,19000,Married,1,1,Yes
1,8,52500,24,63,female,Higher education,Tourism,0,2,1000,Another,3,1,No
2,4,9500,3,32,male,Higher education,Household appliances,1,2,36000,Another,0,1,No
3,10,52500,6,64,male,Higher education,Furniture,0,2,51000,Another,0,0,No
4,5,8000,12,34,female,Secondary special education,Cell phones,1,2,21000,Another,1,0,No


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,month,category,0.0,0.0,12.0,"11, 12, 10, 3, 7, 8, 1, 2, 9, 4"
1,sex,category,0.0,0.0,2.0,"male, female"
2,education,category,0.0,0.0,6.0,"Secondary special education, Higher education, Secondary education, Incomplete higher education, Incomplete secondary education, PhD degree"
3,product_type,category,0.0,0.0,22.0,"Cell phones, Household appliances, Computers, Furniture, Clothing, Cosmetics and beauty services, Windows & Doors, Tourism, Jewelry, Construction Materials"
4,having_children_flg,category,0.0,0.0,2.0,"0, 1"
5,region,category,0.0,0.0,3.0,"2, 0, 1"
6,family_status,category,0.0,0.0,3.0,"Another, Married, Unmarried"
7,phone_operator,category,0.0,0.0,5.0,"1, 0, 2, 3, 4"
8,is_client,category,0.0,0.0,2.0,"1, 0"
9,bad_customer,category,0.0,0.0,2.0,"No, Yes"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
credit_amount,1723.0,29264.654672,27926.778301,5000.0,301000.0
credit_term,1723.0,11.546721,6.548354,3.0,36.0
age,1723.0,35.911782,13.120203,18.0,90.0
income,1723.0,32652.350551,20913.193158,1000.0,401000.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column              rank                                              
bad_customer        1                                 No   1527  88.62
                    2                                Yes    196  11.38
education           1        Secondary special education    836  48.52
                    2                   Higher education    585  33.95
                    3                Secondary education    208  12.07
                    4        Incomplete higher education     86   4.99
                    5     Incomplete secondary education      5   0.29
family_status       1                            Another   1201  69.70
                    2                            Married    444  25.77
                    3                          Unmarried     78   4.53
having_children_flg 1                                  0    985  57.17
                    2                                  1    738  42.83
is_client           1                                  1   1042  60.48
                    2                                  0    681  39.52
month               1                                 11    174  10.10
                    2                                 12    162   9.40
                    3                                 10    160   9.29
                    4                                  3    158   9.17
                    5                                  7    145   8.42
phone_operator      1                                  1    666  38.65
                    2                                  0    536  31.11
                    3                                  2    317  18.40
                    4                                  3    177  10.27
                    5                                  4     27   1.57
product_type        1                        Cell phones    498  28.90
                    2               Household appliances    471  27.34
                    3                          Computers    178  10.33
                    4                          Furniture    164   9.52
                    5                           Clothing     88   5.11
region              1                                  2   1414  82.07
                    2                                  0    240  13.93
                    3                                  1     69   4.00
sex                 1                               male    931  54.03
                    2                             female    792  45.97

In [8]:
# Target Distribution
target_df

,count,pct
bad_customer,,
No,1527,88.62
Yes,196,11.38


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)


splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019d2569-ca61-7ed6-97da-29de641eb79d
2ad61aba93b606cc7a11b6d127b455c4522f5585e0cf259422a64092bf6b9619
